# DuckDB playground — the feature store

A scratchpad for exploring the A4 Parquet feature store through DuckDB (the query layer).
Prerequisite: build the store once with
`uv run python -m src.ingest.feature_store`.

Everything here reads the on-disk Parquet directly — no Spark. DuckDB pushes filters and
column projection down into the Parquet files, so slice-reads over 59M rows stay fast.

In [1]:
import sys
from pathlib import Path

import duckdb

# Make `src` importable regardless of where the kernel's cwd is.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.ingest.feature_store import FEATURE_STORE_DIR  # noqa: E402

GLOB = str(FEATURE_STORE_DIR / "**" / "*.parquet")
con = duckdb.connect()  # in-memory; the data lives in Parquet on disk
# A view over the whole store. hive_partitioning=true recovers store_id from the dir names.
con.execute(
    f"CREATE VIEW fs AS SELECT * FROM read_parquet('{GLOB}', hive_partitioning=true)"
)


def q(sql: str):
    """Run SQL against the store view and return a pandas DataFrame."""
    return con.execute(sql).df()


print("feature store:", FEATURE_STORE_DIR)
q("SELECT COUNT(*) AS rows, COUNT(DISTINCT id) AS series, COUNT(DISTINCT store_id) AS stores FROM fs")

feature store: /Users/emdomingo/repositories/retail-demand-forecasting/data/processed/feature_store


,rows,series,stores
0,59181090,30490,10


## Partitions — one directory per store
Because the store is partitioned by `store_id`, filtering on it prunes to a single
directory (DuckDB skips the other nine entirely).

In [2]:
q("SELECT store_id, COUNT(*) AS rows FROM fs GROUP BY store_id ORDER BY store_id")

,store_id,rows
0,CA_1,5918109
1,CA_2,5918109
2,CA_3,5918109
3,CA_4,5918109
4,TX_1,5918109
5,TX_2,5918109
6,TX_3,5918109
7,WI_1,5918109
8,WI_2,5918109
9,WI_3,5918109


## A single series, ordered by date
**Always `ORDER BY date`** — DuckDB doesn't guarantee row order, and an unordered pull
would scramble the lags/rolling means. Note the warm-up nulls (`lag_7` for the first week).

In [3]:
q("""
    SELECT *
    FROM fs
    ORDER BY date
    LIMIT 14
""")

,id,item_id,dept_id,cat_id,state_id,date,d_int,wm_yr_wk,wday,month,...,rmean_7,rmean_28,event_name_1,event_type_1,is_event,snap,sell_price,has_price,price_change_pct,store_id
0,FOODS_1_126_CA_1_evaluation,FOODS_1_126,FOODS_1,FOODS,CA,2011-01-29,1,11101,1,1,...,NaN,NaN,None,None,0,0,NaN,0,NaN,CA_1
1,FOODS_1_152_CA_1_evaluation,FOODS_1_152,FOODS_1,FOODS,CA,2011-01-29,1,11101,1,1,...,NaN,NaN,None,None,0,0,6.98,1,NaN,CA_1
2,FOODS_1_173_CA_1_evaluation,FOODS_1_173,FOODS_1,FOODS,CA,2011-01-29,1,11101,1,1,...,NaN,NaN,None,None,0,0,1.97,1,NaN,CA_1
3,FOODS_1_212_CA_1_evaluation,FOODS_1_212,FOODS_1,FOODS,CA,2011-01-29,1,11101,1,1,...,NaN,NaN,None,None,0,0,NaN,0,NaN,CA_1
4,FOODS_2_075_CA_1_evaluation,FOODS_2_075,FOODS_2,FOODS,CA,2011-01-29,1,11101,1,1,...,NaN,NaN,None,None,0,0,NaN,0,NaN,CA_1
5,FOODS_1_153_CA_1_evaluation,FOODS_1_153,FOODS_1,FOODS,CA,2011-01-29,1,11101,1,1,...,NaN,NaN,None,None,0,0,1.68,1,NaN,CA_1
6,FOODS_1_205_CA_1_evaluation,FOODS_1_205,FOODS_1,FOODS,CA,2011-01-29,1,11101,1,1,...,NaN,NaN,None,None,0,0,NaN,0,NaN,CA_1
7,FOODS_2_006_CA_1_evaluation,FOODS_2_006,FOODS_2,FOODS,CA,2011-01-29,1,11101,1,1,...,NaN,NaN,None,None,0,0,NaN,0,NaN,CA_1
8,FOODS_2_048_CA_1_evaluation,FOODS_2_048,FOODS_2,FOODS,CA,2011-01-29,1,11101,1,1,...,NaN,NaN,None,None,0,0,NaN,0,NaN,CA_1
9,FOODS_2_051_CA_1_evaluation,FOODS_2_051,FOODS_2,FOODS,CA,2011-01-29,1,11101,1,1,...,NaN,NaN,None,None,0,0,2.50,1,NaN,CA_1


In [4]:
q("""
    SELECT date, sales, lag_7, rmean_7, snap, sell_price
    FROM fs
    WHERE id = 'FOODS_3_090_CA_3_evaluation'
    ORDER BY date
    LIMIT 14
""")

,date,sales,lag_7,rmean_7,snap,sell_price
0,2011-01-29,108,<NA>,NaN,0,1.25
1,2011-01-30,132,<NA>,108.000000,0,1.25
2,2011-01-31,102,<NA>,120.000000,0,1.25
3,2011-02-01,120,<NA>,114.000000,1,1.25
4,2011-02-02,106,<NA>,115.500000,1,1.25
5,2011-02-03,123,<NA>,113.599998,1,1.25
6,2011-02-04,279,<NA>,115.166664,1,1.25
7,2011-02-05,175,108,138.571426,1,1.25
8,2011-02-06,186,132,148.142853,1,1.25
9,2011-02-07,120,102,155.857147,1,1.25


The same read via the project helper (identical result, with the ORDER BY baked in):

In [5]:
from src.query.slices import read_store_slice  # noqa: E402

read_store_slice("CA_3", columns=["id", "date", "sales", "lag_7", "rmean_7"]).head(10)

,id,date,sales,lag_7,rmean_7
0,FOODS_1_001_CA_3_evaluation,2011-01-29,1,<NA>,NaN
1,FOODS_1_001_CA_3_evaluation,2011-01-30,2,<NA>,1.000000
2,FOODS_1_001_CA_3_evaluation,2011-01-31,1,<NA>,1.500000
3,FOODS_1_001_CA_3_evaluation,2011-02-01,1,<NA>,1.333333
4,FOODS_1_001_CA_3_evaluation,2011-02-02,1,<NA>,1.250000
5,FOODS_1_001_CA_3_evaluation,2011-02-03,2,<NA>,1.200000
6,FOODS_1_001_CA_3_evaluation,2011-02-04,0,<NA>,1.333333
7,FOODS_1_001_CA_3_evaluation,2011-02-05,1,1,1.142857
8,FOODS_1_001_CA_3_evaluation,2011-02-06,1,2,1.142857
9,FOODS_1_001_CA_3_evaluation,2011-02-07,1,1,1.000000


## Segment rollups (a taste of the D2 dashboard)
Aggregate accuracy hides spiky high-value items — segment-level views are the point.
Total units by category, and by department within a store.

In [6]:
q("""
    SELECT cat_id, SUM(sales) AS units, ROUND(AVG(sales), 2) AS avg_daily
    FROM fs
    GROUP BY cat_id
    ORDER BY units DESC
""")

,cat_id,units,avg_daily
0,FOODS,45922427.0,1.65
1,HOUSEHOLD,14764090.0,0.73
2,HOBBIES,6240656.0,0.57


In [7]:
q("""
    SELECT dept_id, SUM(sales) AS units
    FROM fs
    WHERE store_id = 'CA_3'
    GROUP BY dept_id
    ORDER BY units DESC
""")

,dept_id,units
0,FOODS_3,5653729.0
1,HOUSEHOLD_1,2185551.0
2,FOODS_2,1221960.0
3,HOBBIES_1,900072.0
4,FOODS_1,749971.0
5,HOUSEHOLD_2,574716.0
6,HOBBIES_2,77541.0


## SNAP effect — a first look (the C3 causal question)
Average daily units on SNAP vs non-SNAP days, for FOODS in California. A raw gap here is
suggestive, not causal — Feature C does the real identification.

In [8]:
q("""
    SELECT snap, ROUND(AVG(sales), 3) AS avg_units, COUNT(*) AS obs
    FROM fs
    WHERE state_id = 'CA' AND cat_id = 'FOODS'
    GROUP BY snap
    ORDER BY snap
""")

,snap,avg_units,obs
0,0,1.694,7478148
1,1,1.867,3678720


## Events vs ordinary days
Average units on event days vs non-event days (events can suppress *or* lift demand —
e.g. Christmas is a store closure).

In [9]:
q("""
    SELECT is_event, ROUND(AVG(sales), 3) AS avg_units, COUNT(*) AS obs
    FROM fs
    WHERE store_id = 'CA_3'
    GROUP BY is_event
    ORDER BY is_event
""")

,is_event,avg_units,obs
0,0,1.927,5436367
1,1,1.844,481742


## Price changes — where promos show up
Weeks with a real price cut (`price_change_pct` well below 0). The data-dictionary warns
deep cuts near −1.0 are penny-price artifacts; the clean promo band is roughly −0.15 to −0.45.

In [10]:
q("""
    SELECT id, date, sell_price, ROUND(price_change_pct, 3) AS pct_change
    FROM fs
    WHERE store_id = 'CA_3'
      AND price_change_pct BETWEEN -0.45 AND -0.15
    ORDER BY price_change_pct
    LIMIT 15
""")

,id,date,sell_price,pct_change
0,HOUSEHOLD_2_462_CA_3_evaluation,2013-08-31,4.67,-0.449
1,HOUSEHOLD_2_462_CA_3_evaluation,2013-09-01,4.67,-0.449
2,HOUSEHOLD_2_462_CA_3_evaluation,2013-09-02,4.67,-0.449
3,HOUSEHOLD_2_462_CA_3_evaluation,2013-09-03,4.67,-0.449
4,HOUSEHOLD_2_462_CA_3_evaluation,2013-09-04,4.67,-0.449
5,HOUSEHOLD_2_462_CA_3_evaluation,2013-09-05,4.67,-0.449
6,HOUSEHOLD_2_462_CA_3_evaluation,2013-09-06,4.67,-0.449
7,HOBBIES_1_317_CA_3_evaluation,2012-05-19,3.98,-0.446
8,HOBBIES_1_317_CA_3_evaluation,2012-05-20,3.98,-0.446
9,HOBBIES_1_317_CA_3_evaluation,2012-05-21,3.98,-0.446


## Your turn
The `fs` view and the `q("...")` helper are ready. A few ideas:
- weekly seasonality: `AVG(sales)` grouped by `wday`
- structural vs demand zeros: compare `sales = 0` rows where `has_price = 1` vs `0`
- a specific item's price history across stores

In [11]:
# scratch — write your own query
q("SELECT wday, ROUND(AVG(sales), 3) AS avg_units FROM fs GROUP BY wday ORDER BY wday")

,wday,avg_units
0,1,1.368
1,2,1.355
2,3,1.082
3,4,1.000
4,5,0.988
5,6,0.994
6,7,1.127
